<a href="https://colab.research.google.com/github/vyunus/Limpieza-de-datos-bencinera-2022/blob/main/Bencinera_marzo_2022.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

df=pd.read_csv('transacciones_bencinera.csv')
df

In [ ]:
#eliminamos las columnas no utilizadas
df = df.drop(columns=['id_transaccion','id_equipo','pump_site_id','tank_site_id','user_site_id',
                      'volumen_inicial','volumen_final','temp_inicial','temp_final','volumen_comp_15_inicial',
                      'volumen_comp_15_final','codigo_producto','geo_latitud','geo_longitud','id_industria'])

In [ ]:
'Columnas eliminadas'
#id_transaccion: se elimina ya que existe un identificador unico unique_id_transa
#id_equipo: este identificador no es relevante para el trabajo
#pump_site_id: se elimina, ya que la información también la tenemos en id_bomba
#tank_site_id: se elimina, porque no es necesario para nuestro analisis
#user_site_id: tiene demasiados datos faltantes y es innecesario
#volumen_inicial: es el volumen de combustible con el que esta cargado al llegar (es innecesario)
#volumen_final: volumen de combustible al finalizar la carga
#temp_inicial: es la temperatura de combustible al llegar (es innecesario)
#temp_final: temperatura de combustible al finalizar la carga
#volumen_comp_15_inicial: no es relevante para el analisis
#volumen_comp_15_final: no es relevante para el analisis
#codigo_producto: ya existe la columna nombre_prod
#geo_latitud: ubicacion precisa (no lo usaremos)
#geo_longitud: ubicacion precisa (no lo usaremos)
#id_industria: ya sabemos a que industria pertenece en la columna industria

In [ ]:
#revisamos que se hayan eliminado correctamente
df.columns

In [ ]:
#check valores duplicados
dup_total = df.duplicated().sum()

In [ ]:

print("\nRegistros detectados como duplicados:")
print(df[df.duplicated(keep=False)])

In [ ]:
# Limpiamos los datos duplicados
df = df.drop_duplicates().copy()
print("\nLimpieza realizada. Nuevo total de filas:", len(df))

In [ ]:
# Identificamos cuántos datos nulos tenemos en las columnas
nulos_por_columna = df.isnull().sum()
print("Conteo de datos faltantes por columna:")
print(nulos_por_columna)

In [ ]:
print(round(df['producto'].describe()),2)
#Revisada con describe() para ver cuartiles, media, maximos y minimos

In [ ]:
df['producto'] = pd.to_numeric(df['producto'], errors='coerce')
#Transformacion a numericos, sino se marca como nulo

In [ ]:
df['producto'] = df['producto'].fillna(1) #de nulo a 1 para tener mejor manejo de los datos

In [ ]:
import pandas as pd

print('---LIMPIEZA---')

# trabajamos una lista especifica con valores cerrados de el 0 al 4
df['producto'] = pd.to_numeric(df['producto'], errors='coerce')
df['producto'] = df['producto'].where(df['producto'].isin([0, 1, 2, 3, 4]))#Check de si se cumple dicha condicion (0 a 4)

# rellenamos los datos nulos con la moda
moda_producto = df['producto'].mode()
df['producto'] = df['producto'].fillna(moda_producto) #Se trabaja con la moda en los nulos
print("Columna producto a numero entero:")
print(df['producto'].value_counts())

In [ ]:
print("\n--- REPORTE DE NULOS DE PRODUCTO ---")
print(df['producto'].isnull().sum()) #Check matutino de nulos

In [ ]:
#Calculamos la mediana de la columna cantidad al pertenecer al 50% de los datos
mediana_cantidad = df['cantidad'].median()

# Rellenamos los valores nulos con la mediana
df['cantidad'] = df['cantidad'].fillna(mediana_cantidad)

# tambien se limpia la cant_comp_15 y sus valores nulos se sustituyen por la mediana
df['cantidad_comp_15'] = df['cantidad_comp_15'].fillna(df['cantidad_comp_15'].median())

print(f"{mediana_cantidad}")

In [ ]:
nulos_cantidad = df['cantidad'].isnull().sum()
print(f"Nulos encontrados en cantidad: {nulos_cantidad}")#Check de nulos en cantidad

In [ ]:
import numpy as np
#Se utiliza el rango intercuartilico de manera que podamos determinar tanto los valores extremos como los minimos


# calculamos el rango normal (IQR: sacando Q1 y Q3 para definir el Límite Superior)
Q1_cant = df['cantidad'].quantile(0.25)
Q3_cant = df['cantidad'].quantile(0.75)
limite_sup_cant = Q3_cant + 1.5 * (Q3_cant - Q1_cant)

Q1_comp = df['cantidad_comp_15'].quantile(0.25)
Q3_comp = df['cantidad_comp_15'].quantile(0.75)
limite_sup_comp = Q3_comp + 1.5 * (Q3_comp - Q1_comp)

# filtramos los numeros grandes usando np.where como un condicional rápido
#se consulta una fila y en caso de superar el limite maximo, lo remplazamos por la mediana. En caso de ser un valor normal se queda igual
df['cantidad'] = np.where(df['cantidad'] > limite_sup_cant, df['cantidad'].median(), df['cantidad'])
df['cantidad_comp_15'] = np.where(df['cantidad_comp_15'] > limite_sup_comp, df['cantidad_comp_15'].median(), df['cantidad_comp_15'])


# Estos numeros atipicos se reemplazan al igual que el anterior, por la mediana
df['cantidad'] = np.where(df['cantidad'] <= 0, df['cantidad'].median(), df['cantidad'])
df['cantidad_comp_15'] = np.where(df['cantidad_comp_15'] <= 0, df['cantidad_comp_15'].median(), df['cantidad_comp_15'])


print(df[['cantidad', 'cantidad_comp_15']].describe().round(2))

In [ ]:
# Identificamos nuevamente cuantos   datos nulos tenemos en las columnas
nulos_por_columna = df.isnull().sum()
print("Conteo de datos faltantes por columna:")
print(nulos_por_columna)

In [ ]:
# Reemplazo de nulos siguiento el formato del codigo error
df['codigo_error'].fillna('NA')

In [ ]:
df['codigo_error'] = df['codigo_error'].fillna('NA')
print(df.isnull().sum())

In [ ]:
#limpieza caracter basura industria
df['industria'] = df['industria'].str.replace(r'[^a-zA-Z0-9 ]', '', regex=True)#Formateo de los caracteres basura
print(df['industria'].head(10))

In [ ]:
df['industria'].unique().tolist() #Se revisa que esten todas las industrias validas

In [ ]:
# Rellenamos los nulos de industria creamos una nueva categoria llamada desconocido
df['industria'] = df['industria'].fillna('Desconocido')

# Verificamos si queda algun nulo en industria
print(f"Nulos restantes en industria: {df['industria'].isnull().sum()}")


print(df['industria'].value_counts().head())

In [ ]:
import pandas as pd
import numpy as np
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df['timestamp_stop'] = pd.to_datetime(df['timestamp_stop'], errors='coerce')

In [ ]:
 #restamoos las fechas para calcular el tiempo de carga
df['duracion_carga_seg'] = (df['timestamp_stop'] - df['timestamp']).dt.total_seconds()

In [ ]:
#Lunes=0, Domingo=6
df['dia_de_semana'] = df['timestamp'].dt.dayofweek

In [ ]:
# eliminamos estas columnas ya que extrajimos su info para crear las nuevas
df = df.drop(columns=['timestamp', 'timestamp_stop'])

In [ ]:
#se rellenan las fechas vacias
df['duracion_carga_seg'] = df['duracion_carga_seg'].fillna(df['duracion_carga_seg'].median())
df['dia_de_semana'] = df['dia_de_semana'].fillna(df['dia_de_semana'].mode()[0])

In [ ]:

print(df[['duracion_carga_seg', 'dia_de_semana']].head())

In [ ]:
# Identificamos cuantos datos nulos tenemos en las columnas
nulos_por_columna = df.isnull().sum()
print("Conteo de datos faltantes por columna:")
print(nulos_por_columna)

In [ ]:
#al crear un "diccionario" o mapa de datos completos nos permite agrupar el codigo producto por nombre producto
mapa_productos = df.dropna(subset=['nombre_prod']).set_index('producto')['nombre_prod'].to_dict()

# se puede ver cual codigo producto pertenece al nombre producto
print("Mapa de productos descubierto:", mapa_productos)

In [ ]:

# con el mapa creado se rellenan los nombre_prod vacios basandose en su codigo
df['nombre_prod'] = df['nombre_prod'].fillna(df['producto'].map(mapa_productos))

# Verificamos cuantos nulos logramos salvar
nulos_restantes = df['nombre_prod'].isnull().sum()
print(f"\nNulos restantes en nombre_prod después del rescate inteligente: {nulos_restantes}")

In [ ]:
# auqellos que no fueron encontrados, los dejaremos con el nombre producto sin registro
df['nombre_prod'] = df['nombre_prod'].fillna('Producto Sin Registro')

In [ ]:
filas_parchadas = df[df['nombre_prod'] == 'Producto Sin Registro']

print(f"Hay {len(filas_parchadas)} filas con este parche.")


print(filas_parchadas[['producto', 'nombre_prod', 'cantidad']].head())

In [ ]:
#Originalmente se iba a trabajar con la columna pero al tener poca riqueza en datos se descarta
df = df.drop(columns=['veh_efficiency'])

In [ ]:
nulos_por_columna = df.isnull().sum()
print("Conteo de datos faltantes por columna:")
print(nulos_por_columna)

In [ ]:
# 1. Las Importaciones (Nuevas Herramientas de Control)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Definimos cuáles son nuestras columnas numéricas y categóricas
columnas_numericas = ['cantidad', 'cantidad_comp_15']
columnas_categoricas = ['industria', 'nombre_prod', 'producto', 'codigo_error']

# 2. Definición del Procesador (El "Plan Maestro")
preprocesador = ColumnTransformer(transformers=[
    ('num', StandardScaler(), columnas_numericas), # Estándar: Media 0, Desv 1
    ('cat', OneHotEncoder(handle_unknown='ignore'), columnas_categoricas) # Crea columnas binarias
])

# 3.(Pipeline)
pipeline_final = Pipeline(steps=[('prep', preprocesador)])

# 4. Ejecución Mágica: fit_transform
# Resultado: Una matriz lista para la Inteligencia Artificial
datos_listos = pipeline_final.fit_transform(df)


print("Tamaño de la matriz final transformada (filas y columnas)")
print(datos_listos.shape)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

print("--- INICIANDO TRANSFORMACIÓN DEL DATASET DE LA GASOLINERA ---")

# 1. Separamos las columnas por tipo
columnas_numericas = ['cantidad', 'cantidad_comp_15']
columnas_categoricas = ['industria', 'nombre_prod', 'producto', 'codigo_error']

# 2. Definimos el Procesador
preprocesador = ColumnTransformer(transformers=[
    ('num', StandardScaler(), columnas_numericas),
    ('cat', OneHotEncoder(handle_unknown='ignore'), columnas_categoricas)
])

# 3. Creamos el Pipeline
pipeline_final = Pipeline(steps=[('prep', preprocesador)])

# 4. Transformacion final a Matriz numerica
matriz_ia = pipeline_final.fit_transform(df)


print(f"Matriz lista para Machine Learning: {matriz_ia.shape}")

In [ ]:
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

print("--- 2. TRANSFORMACIÓN MANUAL  ---")

# Creamos una copia para no alterar nuestro dataset limpio original
df_manual = df.copy()

# A. Label Encoding (Palabras a numeros secuenciales)
# Convertimos las palabras de 'industria' en números mineria=0, Transporte=1
le = LabelEncoder()
df_manual['industria_ID'] = le.fit_transform(df_manual['industria'].astype(str))

# Min- Max Scaling (Escalar la cantidad de litros entre 0 y 1)
# La carga mas pequeña de la historia sera 0, y la más grande sera 1.
mm_scaler = MinMaxScaler()
df_manual['cantidad_Norm'] = mm_scaler.fit_transform(df_manual[['cantidad']])

# Comprobación Visual
print("\nComparación: Original vs Transformado")
# Mostramos una muestra de como se ven las columnas viejas junto a las nuevas
print(df_manual[['industria', 'industria_ID', 'cantidad', 'cantidad_Norm']].head(10))
print("-" * 50)

In [ ]:
df_manual[['industria', 'industria_ID', 'cantidad', 'cantidad_Norm']].head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))

df.groupby('industria')['cantidad_comp_15'].mean().sort_values().plot(kind='bar', ax=ax)

ax.set_ylabel("Cantidad promedio de combustible")
ax.set_xlabel("Industria")
ax.set_title("Promedio de combustible cargado por industria")

ax.grid(linestyle=":")

plt.tight_layout()
plt.show()